<a href="https://colab.research.google.com/github/aerofa45/tb-guard-multimodal-ai/blob/main/TBGuard_4_App_RAG_LangGraph_Gradio_Last.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TB-Guard Notebook 4 — UPDATED FINAL

Updated for the fixed Notebook 2 clinical model and the WHO rule-based Notebook 3 genomic expert.

In [1]:
!pip install -q gradio langchain langchain-core langchain-community langchain-openai langchain-google-genai langchain-anthropic langchain-huggingface langgraph faiss-cpu sentence-transformers torch torchvision joblib pillow pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires request

In [2]:
import os, json, shutil, time, traceback
from datetime import datetime
from pathlib import Path
from typing import TypedDict, Dict, Any, List, Optional
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torchvision.transforms as T
import gradio as gr
from PIL import Image
from torchvision import models
from google.colab import files
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, END

BASE=Path('tbguard_project')
MODELS=BASE/'artifacts/models'
RAG_SOURCE_DIR=BASE/'data/rag_sources'
RAG_DIR=BASE/'artifacts/rag_faiss'
DEMO_DIR=BASE/'demo_cases'
LOG_DIR=BASE/'logs'
EXPORT_DIR=BASE/'exports'
for d in [MODELS,RAG_SOURCE_DIR,RAG_DIR,DEMO_DIR,LOG_DIR,EXPORT_DIR]: d.mkdir(parents=True, exist_ok=True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


## 1. Upload artifacts

In [ ]:
print("Upload required artifacts.")
print("You may upload either individual files or ZIP files containing them.")
print("Required: cxr_efficientnet_b0_tb.pt, clinical_tb_model.joblib, genomic_resistance_model.joblib")
print("Optional but recommended: who_mutation_catalogue_rag_text.txt")

from google.colab import files
from pathlib import Path
import shutil
import zipfile
import os

uploaded = files.upload()

UPLOADS_DIR = BASE / "uploads"
UPLOADS_DIR.mkdir(parents=True, exist_ok=True)

# Save uploads first
for name in uploaded.keys():
    src = Path(name)
    dest = UPLOADS_DIR / name
    shutil.move(str(src), str(dest))
    print("Uploaded saved to:", dest)

    # If zip, extract it
    if dest.suffix.lower() == ".zip":
        extract_dir = UPLOADS_DIR / dest.stem
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(dest, "r") as z:
            z.extractall(extract_dir)

        print("Extracted ZIP to:", extract_dir)

# Recursively search all uploads for needed files
required_model_files = [
    "cxr_efficientnet_b0_tb.pt",
    "clinical_tb_model.joblib",
    "genomic_resistance_model.joblib"
]

optional_rag_files = [
    "who_mutation_catalogue_rag_text.txt"
]

# Helper: find file by exact name anywhere under uploads
def find_file_recursive(root, filename):
    matches = list(Path(root).rglob(filename))
    return matches[0] if matches else None

# Move required model files into MODELS folder
for fname in required_model_files:
    found = find_file_recursive(UPLOADS_DIR, fname)

    if found is None:
        raise FileNotFoundError(
            f"Could not find required file: {fname}. "
            f"Check that it exists inside the uploaded ZIP or upload it directly."
        )

    dest = MODELS / fname
    shutil.copy2(found, dest)
    print("Model artifact ready:", dest)

# Move optional WHO RAG text into RAG source folder
for fname in optional_rag_files:
    found = find_file_recursive(UPLOADS_DIR, fname)

    if found is not None:
        dest = RAG_SOURCE_DIR / fname
        shutil.copy2(found, dest)
        print("Optional RAG source ready:", dest)
    else:
        print("Optional file not found:", fname)

# Also copy demo images if present
demo_image_dirs = list(UPLOADS_DIR.rglob("demo_images"))

if demo_image_dirs:
    demo_src = demo_image_dirs[0]
    demo_dest = BASE / "demo_images"

    if demo_dest.exists():
        shutil.rmtree(demo_dest)

    shutil.copytree(demo_src, demo_dest)
    print("Demo images copied to:", demo_dest)
else:
    print("No demo_images folder found. You may need to upload 3 X-ray images separately if demo cases require them.")

# Final verification
for fname in required_model_files:
    path = MODELS / fname
    print(fname, "exists:", path.exists(), "size:", path.stat().st_size if path.exists() else None)

print("Artifacts ready.")

Upload required artifacts.
You may upload either individual files or ZIP files containing them.
Required: cxr_efficientnet_b0_tb.pt, clinical_tb_model.joblib, genomic_resistance_model.joblib
Optional but recommended: who_mutation_catalogue_rag_text.txt


## 2. API keys

In [ ]:
def load_api_keys_from_colab_secrets():
    """
    Safely loads API keys from Colab Secrets.
    Keys are not printed and are not stored inside the notebook.
    """
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY") or ""
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") or ""

        print("Colab Secrets checked.")

    except Exception as e:
        print("Colab Secrets not available or not configured:", e)
        os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
        os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")
        os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY", "")

load_api_keys_from_colab_secrets()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

print("OpenAI key loaded:", bool(OPENAI_API_KEY))
print("Gemini key loaded:", bool(GOOGLE_API_KEY))
print("Claude key loaded:", bool(ANTHROPIC_API_KEY))

## 3. Upload 3 demo X-ray images

In [ ]:
uploaded_images=files.upload()
img_paths=[]
for name in uploaded_images.keys():
    dest=DEMO_DIR/name
    shutil.move(name,dest)
    img_paths.append(str(dest))
if not img_paths: raise RuntimeError('Upload at least one X-ray image')
while len(img_paths)<3: img_paths.append(img_paths[0])
print(img_paths[:3])

## 4. Build local FAISS RAG

In [ ]:
base_sources={
 'tb_overview.txt':'Tuberculosis commonly presents with chronic cough, fever, night sweats, fatigue, weight loss, and chest X-ray findings such as upper-lobe infiltrates, consolidation, nodular opacity, or cavitary lesions. Confirmatory testing is required.',
 'drug_sensitive_tb.txt':'Drug-sensitive TB is suspected when clinical symptoms and radiographic findings support TB without evidence of rifampicin or isoniazid resistance.',
 'mdr_tb.txt':'MDR-TB is tuberculosis resistant to at least rifampicin and isoniazid. Rifampicin resistance is an important warning marker requiring drug susceptibility testing.',
 'pre_xdr_tb.txt':'MDR/RR-TB with additional fluoroquinolone resistance raises concern for pre-XDR tuberculosis. Genomic resistance markers require laboratory confirmation.',
 'human_in_loop.txt':'AI systems for tuberculosis detection are decision-support tools only. A qualified physician must confirm diagnosis and treatment decisions.'
}
for fn,txt in base_sources.items():
    p=RAG_SOURCE_DIR/fn
    if not p.exists(): p.write_text(txt,encoding='utf-8')
docs=[]
for p in sorted(RAG_SOURCE_DIR.glob('*.txt')):
    txt=p.read_text(encoding='utf-8', errors='ignore').strip()
    if txt: docs.append(Document(page_content=txt, metadata={'filename':p.name,'source':str(p)}))
print('RAG sources:', [d.metadata['filename'] for d in docs])
chunks=RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=150).split_documents(docs)
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore=FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(str(RAG_DIR))
print('RAG chunks:', len(chunks))

## 5. Load models/artifacts

In [ ]:
IMG_SIZE=224
cxr_tfm=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.ToTensor(),T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def build_cxr_model():
    m=models.efficientnet_b0(weights=None)
    m.classifier[1]=nn.Linear(m.classifier[1].in_features,1)
    return m
ck=torch.load(MODELS/'cxr_efficientnet_b0_tb.pt', map_location=DEVICE)
cxr_model=build_cxr_model().to(DEVICE)
cxr_model.load_state_dict(ck['model_state_dict'])
cxr_model.eval()
cxr_threshold=float(ck.get('threshold', ck.get('best_threshold', ck.get('decision_threshold', 0.5))))
print('CXR threshold:', cxr_threshold)

clinical_artifact=joblib.load(MODELS/'clinical_tb_model.joblib')
clinical_model=clinical_artifact['model']
clinical_features=clinical_artifact['features']
clinical_threshold=float(clinical_artifact.get('threshold',0.5))
clinical_class_mapping=clinical_artifact.get('class_mapping',{0:'Non-TB / not confirmed',1:'TB confirmed/risk'})
print('Clinical features:', clinical_features)
print('Clinical threshold:', clinical_threshold)

genomic_artifact=joblib.load(MODELS/'genomic_resistance_model.joblib')
genomic_features=genomic_artifact.get('features',['rpoB','katG','inhA','gyrA','gyrB','rrs','eis','embB','pncA'])
print('Genomic type:', genomic_artifact.get('model_type'))
print('Genomic features:', genomic_features)

vectorstore=FAISS.load_local(str(RAG_DIR), embeddings, allow_dangerous_deserialization=True)
print('All artifacts loaded.')

# Compatibility checks for latest Notebook 2 and Notebook 3 artifacts
print('\n--- Compatibility checks ---')
required_clinical_keys = ['model', 'features', 'threshold', 'class_mapping']
missing_clinical = [k for k in required_clinical_keys if k not in clinical_artifact]
if missing_clinical:
    raise KeyError(f'clinical_tb_model.joblib is missing keys: {missing_clinical}. Re-run the fixed Notebook 2.')
print('Clinical artifact OK. Saved threshold:', clinical_threshold)
print('Clinical saved metrics:', clinical_artifact.get('metrics', {}))

if genomic_artifact.get('model_type') != 'who_rule_based_catalogue_expert':
    raise ValueError('genomic_resistance_model.joblib is not the WHO rule-based artifact from the latest Notebook 3.')
required_genomic_keys = ['features', 'rule_map', 'class_mapping']
missing_genomic = [k for k in required_genomic_keys if k not in genomic_artifact]
if missing_genomic:
    raise KeyError(f'genomic_resistance_model.joblib is missing keys: {missing_genomic}. Re-run the fixed Notebook 3.')
print('WHO genomic artifact OK.')
if any('XDR-style' in str(v) for v in genomic_artifact.get('class_mapping', {}).values()):
    print('Warning: genomic class_mapping still contains old XDR-style wording. Re-run the updated Notebook 3 Section 5 if you want safer wording.')


## 6. Prediction + retrieval functions

In [ ]:
def predict_cxr(image_path: str):
    x=cxr_tfm(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): prob=torch.sigmoid(cxr_model(x).squeeze()).item()
    pred=int(prob>=cxr_threshold)
    return {'model_name':'EfficientNet-B0 CXR TB classifier','image_path':str(image_path),'tb_probability':round(float(prob),4),'threshold':round(cxr_threshold,4),'predicted_class_id':pred,'predicted_class':'TB' if pred else 'Normal/Low TB','label':'X-ray supports TB suspicion' if pred else 'X-ray does not strongly support TB','confidence':'high' if prob>=cxr_threshold+0.20 or prob<=max(cxr_threshold-0.25,0) else 'medium' if pred else 'low'}

def predict_clinical_tb(clinical_data: Dict[str,Any]):
    row=pd.DataFrame([clinical_data])
    for col in clinical_features:
        if col not in row.columns: row[col]=np.nan
    row=row[clinical_features]
    prob=float(clinical_model.predict_proba(row)[:,1][0])
    pred=int(prob>=clinical_threshold)
    return {'model_name':'Clinical TB risk model','clinical_tb_risk':round(prob,4),'threshold':round(clinical_threshold,4),'predicted_class_id':pred,'predicted_class':clinical_class_mapping.get(pred,str(pred)),'label':'Clinical profile supports TB risk' if pred else 'Clinical profile does not strongly support TB risk','confidence':'high' if prob>=clinical_threshold+0.20 or prob<=max(clinical_threshold-0.25,0) else 'medium' if pred else 'low','features_used':clinical_features,'model_metrics':clinical_artifact.get('metrics',{})}

def predict_genomic_resistance_who_rules(genomic_data: Dict[str,Any]):
    art=genomic_artifact
    rule_map=art.get('rule_map',{})
    features=art.get('features',genomic_features)
    fallback={
        'rpoB':('Rifampicin','rpoB marker suggests rifampicin resistance concern.','Rifampicin resistance is a major marker for RR-TB and possible MDR-TB evaluation.'),
        'katG':('Isoniazid','katG marker suggests isoniazid resistance concern.','Isoniazid resistance combined with rifampicin resistance supports MDR-TB concern.'),
        'inhA':('Isoniazid','inhA marker suggests isoniazid resistance concern.','inhA-associated resistance can contribute to isoniazid resistance.'),
        'gyrA':('Fluoroquinolones','gyrA marker suggests fluoroquinolone resistance concern.','Fluoroquinolone resistance in MDR/RR-TB raises pre-XDR concern.'),
        'gyrB':('Fluoroquinolones','gyrB marker suggests fluoroquinolone resistance concern.','Fluoroquinolone resistance in MDR/RR-TB raises pre-XDR concern.'),
        'rrs':('Second-line injectable drugs','rrs marker suggests second-line injectable resistance concern.','Second-line resistance requires specialist review and drug susceptibility testing.'),
        'eis':('Kanamycin / injectable-associated resistance','eis marker suggests injectable-associated resistance concern.','Injectable-associated resistance should be interpreted with confirmatory testing.'),
        'embB':('Ethambutol','embB marker suggests ethambutol resistance concern.','Ethambutol resistance may affect treatment selection.'),
        'pncA':('Pyrazinamide','pncA marker suggests pyrazinamide resistance concern.','Pyrazinamide resistance may affect treatment selection.')
    }
    markers={}
    for g in features:
        try: markers[g]=int(genomic_data.get(g,0))
        except Exception: markers[g]=0
    rif=markers.get('rpoB',0)==1
    inh=markers.get('katG',0)==1 or markers.get('inhA',0)==1
    fq=markers.get('gyrA',0)==1 or markers.get('gyrB',0)==1
    inj=markers.get('rrs',0)==1 or markers.get('eis',0)==1
    other=markers.get('embB',0)==1 or markers.get('pncA',0)==1
    detected=[g for g,v in markers.items() if v==1]
    evidence=[]
    for g in detected:
        info=rule_map.get(g,{})
        if not info or not info.get('drug') or not info.get('interpretation'):
            drug,interp,role=fallback.get(g,('Unknown',f'{g} marker detected; clinical review required.','Requires confirmatory DST.'))
            info={'drug':drug,'interpretation':interp,'clinical_role':role}
        evidence.append({'gene':g,'drug':info.get('drug','Unknown'),'interpretation':info.get('interpretation',f'{g} marker detected.'),'clinical_role':info.get('clinical_role','Requires clinical interpretation and confirmatory DST.')})
    if rif and inh and fq:
        class_id=2; resistance_class='Possible pre-XDR TB concern / MDR-TB with fluoroquinolone resistance'; confidence='high'
    elif rif and inh:
        class_id=1; resistance_class='Possible MDR-TB concern'; confidence='high'
    elif rif:
        class_id=1; resistance_class='Rifampicin-resistant TB concern; evaluate for MDR-TB'; confidence='medium'
    elif inh or fq or inj or other:
        class_id=1; resistance_class='Possible drug-resistance concern'; confidence='medium'
    else:
        class_id=0; resistance_class='No major genomic resistance marker detected'; confidence='medium'
    class_mapping=art.get('class_mapping',{0:'No major resistance marker detected',1:'Drug-resistant TB / MDR-TB concern',2:'pre-XDR concern / MDR-TB with fluoroquinolone resistance'})
    scores={class_mapping[0]:0.85 if class_id==0 else 0.10, class_mapping[1]:0.85 if class_id==1 else 0.10, class_mapping[2]:0.85 if class_id==2 else 0.10}
    return {'model_name':art.get('model_name','WHO-catalogue-style genomic resistance expert'),'model_type':art.get('model_type','who_rule_based_catalogue_expert'),'predicted_class_id':class_id,'resistance_class':resistance_class,'confidence':confidence,'markers_detected':detected,'resistance_evidence':evidence,'interpreted_resistance':{'rifampicin':rif,'isoniazid':inh,'fluoroquinolone':fq,'second_line_injectable':inj,'ethambutol_or_pyrazinamide':other},'class_probabilities':scores,'probability_note':art.get('probability_note','Rule-based confidence scores, not calibrated ML probabilities.'),'source_basis':art.get('source','WHO mutation catalogue concept.'),'human_in_loop_note':'Genomic resistance interpretation must be confirmed by clinicians and laboratory testing.','features_used':features}

def retrieve_tb_evidence(query: str, k:int=4):
    docs=vectorstore.similarity_search(query,k=k)
    return [{'content':d.page_content,'source':d.metadata.get('filename',d.metadata.get('source','unknown')),'metadata':d.metadata} for d in docs]

## 7. Sanity checks

In [ ]:
sample_clinical={'age':45,'sex':'Male','test_method':'Molecular rapid test','hiv_education':'Yes','district':'Tugu','report_month':11,'report_quarter':4}
print('Clinical sample:')
print(json.dumps(predict_clinical_tb(sample_clinical), indent=2))
for name,sample in {'no_resistance':{'rpoB':0,'katG':0,'inhA':0,'gyrA':0,'gyrB':0,'rrs':0,'eis':0,'embB':0,'pncA':0}, 'mdr':{'rpoB':1,'katG':1,'inhA':0,'gyrA':0,'gyrB':0,'rrs':0,'eis':0,'embB':0,'pncA':0}, 'pre_xdr':{'rpoB':1,'katG':1,'inhA':0,'gyrA':1,'gyrB':0,'rrs':0,'eis':0,'embB':0,'pncA':0}}.items():
    print('\nGenomic', name)
    print(json.dumps(predict_genomic_resistance_who_rules(sample), indent=2))
print('\nCXR sample:')
print(json.dumps(predict_cxr(img_paths[0]), indent=2))

## 8. LLM reviewers

In [ ]:
def fallback_llm_response(agent_name, prompt, reason="API key was not provided or API call failed."):
    return (
        f"[{agent_name} fallback mode]\n"
        f"Reason: {reason}\n\n"
        "Reviewed model outputs and RAG evidence using deterministic fallback logic. "
        "Physician confirmation required."
    )


def gpt_diagnostic_reviewer(patient_id, xray_result, clinical_result, genomic_result, rag_evidence):
    prompt = (
        "Patient " + patient_id +
        "\nX-ray: " + json.dumps(xray_result) +
        "\nClinical: " + json.dumps(clinical_result) +
        "\nGenomic: " + json.dumps(genomic_result) +
        "\nEvidence: " + json.dumps(rag_evidence) +
        "\nGive TB and resistance diagnostic reasoning. Decision support only."
    )

    if not OPENAI_API_KEY:
        return fallback_llm_response("GPT Diagnostic Reviewer", prompt, "OPENAI_API_KEY not loaded.")

    try:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.2,
            api_key=OPENAI_API_KEY
        ).invoke(prompt).content
    except Exception as e:
        return fallback_llm_response("GPT Diagnostic Reviewer", prompt, str(e)[:300])


def gemini_second_opinion_reviewer(patient_id, xray_result, clinical_result, genomic_result, rag_evidence, gpt_opinion):
    prompt = (
        "Challenge/support GPT for " + patient_id +
        "\nXray " + json.dumps(xray_result) +
        "\nClinical " + json.dumps(clinical_result) +
        "\nGenomic " + json.dumps(genomic_result) +
        "\nEvidence " + json.dumps(rag_evidence) +
        "\nGPT " + gpt_opinion
    )

    if not GOOGLE_API_KEY:
        return fallback_llm_response("Gemini Second Opinion Reviewer", prompt, "GOOGLE_API_KEY not loaded.")

    from langchain_google_genai import ChatGoogleGenerativeAI

    # Try cheaper/lighter first to reduce quota pressure.
    candidate_models = [
        "gemini-2.0-flash-lite",
        "gemini-2.0-flash",
        "gemini-2.5-flash-lite",
        "gemini-2.5-flash"
    ]

    last_error = None

    for model_name in candidate_models:
        try:
            return ChatGoogleGenerativeAI(
                model=model_name,
                temperature=0.2,
                google_api_key=GOOGLE_API_KEY
            ).invoke(prompt).content
        except Exception as e:
            last_error = str(e)
            if "RESOURCE_EXHAUSTED" in last_error or "429" in last_error:
                continue
            continue

    return fallback_llm_response(
        "Gemini Second Opinion Reviewer",
        prompt,
        "Gemini quota/model call failed: " + str(last_error)[:300]
    )


def claude_safety_reviewer(patient_id, xray_result, clinical_result, genomic_result, rag_evidence, gpt_opinion, gemini_opinion):
    prompt = (
        "Safety review " + patient_id +
        "\nXray " + json.dumps(xray_result) +
        "\nClinical " + json.dumps(clinical_result) +
        "\nGenomic " + json.dumps(genomic_result) +
        "\nEvidence " + json.dumps(rag_evidence) +
        "\nGPT " + gpt_opinion +
        "\nGemini " + gemini_opinion
    )

    if not ANTHROPIC_API_KEY:
        return fallback_llm_response("Claude Safety Reviewer", prompt, "ANTHROPIC_API_KEY not loaded.")

    try:
        from langchain_anthropic import ChatAnthropic

        return ChatAnthropic(
            model="claude-haiku-4-5",
            temperature=0.2,
            api_key=ANTHROPIC_API_KEY
        ).invoke(prompt).content

    except Exception as e:
        return fallback_llm_response(
            "Claude Safety Reviewer",
            prompt,
            "Anthropic API call failed: " + str(e)[:300]
        )

In [ ]:
# Check if API Works (Sanity Check)

print("Checking LLM reviewer functions...")

dummy_xray = {
    "tb_probability": 0.72,
    "predicted_class": "TB",
    "confidence": "medium"
}

dummy_clinical = {
    "clinical_tb_risk": 0.61,
    "predicted_class": "TB confirmed/risk",
    "confidence": "medium"
}

dummy_genomic = {
    "predicted_class_id": 1,
    "resistance_class": "Possible MDR-TB concern",
    "confidence": "high"
}

dummy_rag = [
    {
        "source": "test_source.txt",
        "content": "TB diagnosis and drug resistance require clinical confirmation."
    }
]

try:
    gpt_test = gpt_diagnostic_reviewer(
        "TEST",
        dummy_xray,
        dummy_clinical,
        dummy_genomic,
        dummy_rag
    )
    print("GPT reviewer OK")
    print(gpt_test[:300])
except Exception as e:
    print("GPT reviewer FAILED:", e)

try:
    gemini_test = gemini_second_opinion_reviewer(
        "TEST",
        dummy_xray,
        dummy_clinical,
        dummy_genomic,
        dummy_rag,
        "GPT says possible TB with resistance concern."
    )
    print("\nGemini reviewer OK")
    print(gemini_test[:300])
except Exception as e:
    print("\nGemini reviewer FAILED:", e)

try:
    claude_test = claude_safety_reviewer(
        "TEST",
        dummy_xray,
        dummy_clinical,
        dummy_genomic,
        dummy_rag,
        "GPT says possible TB.",
        "Gemini agrees but requests confirmation."
    )
    print("\nClaude reviewer OK")
    print(claude_test[:300])
except Exception as e:
    print("\nClaude reviewer FAILED:", e)

print("\nLLM function check completed.")

In [ ]:
for name, response in [
    ("GPT", gpt_test),
    ("Gemini", gemini_test),
    ("Claude", claude_test)
]:
    used_fallback = "fallback mode" in response.lower()
    print(f"{name}: {'FALLBACK USED' if used_fallback else 'REAL API RESPONSE'}")

## 9. LangGraph workflow

In [ ]:
class TBGuardState(TypedDict):
    patient_id:str; xray_image_path:str; clinical_data:Dict[str,Any]; genomic_data:Dict[str,Any]
    xray_result:Dict[str,Any]; clinical_result:Dict[str,Any]; genomic_result:Dict[str,Any]
    rag_query:str; rag_evidence:List[Dict[str,Any]]
    gpt_opinion:str; gemini_opinion:str; claude_safety_opinion:str
    debate_summary:str; judge_decision:Dict[str,Any]; final_report:str
    workflow_logs:List[str]; structured_logs:List[Dict[str,Any]]

def add_log(state,node,status,message,runtime_sec=None,details=None):
    ev={'timestamp':datetime.now().isoformat(timespec='seconds'),'patient_id':state.get('patient_id','unknown'),'node':node,'status':status,'message':message,'runtime_sec':runtime_sec,'details':details or {}}
    state['structured_logs'].append(ev); state['workflow_logs'].append(f"{ev['timestamp']} | {node} | {status}: {message}")

def timed_node(node_name):
    def dec(fn):
        def wrapper(state):
            start=time.time()
            try:
                out=fn(state); add_log(out,node_name,'success','completed',round(time.time()-start,4)); return out
            except Exception as e:
                add_log(state,node_name,'error',str(e),round(time.time()-start,4),{'traceback':traceback.format_exc()}); raise
        return wrapper
    return dec

@timed_node('xray_agent')
def node_xray(s): s['xray_result']=predict_cxr(s['xray_image_path']); return s
@timed_node('clinical_agent')
def node_clinical(s): s['clinical_result']=predict_clinical_tb(s['clinical_data']); return s
@timed_node('genomic_agent')
def node_genomic(s): s['genomic_result']=predict_genomic_resistance_who_rules(s['genomic_data']); return s
@timed_node('rag_retrieval')
def node_rag(s):
    q='Patient '+s['patient_id']+' Xray '+json.dumps(s['xray_result'])+' Clinical '+json.dumps(s['clinical_result'])+' Genomic '+json.dumps(s['genomic_result'])+' Need evidence about TB, MDR, pre-XDR and human confirmation.'
    s['rag_query']=q; s['rag_evidence']=retrieve_tb_evidence(q,4); return s
@timed_node('gpt_reviewer')
def node_gpt(s): s['gpt_opinion']=gpt_diagnostic_reviewer(s['patient_id'],s['xray_result'],s['clinical_result'],s['genomic_result'],s['rag_evidence']); return s
@timed_node('gemini_reviewer')
def node_gemini(s): s['gemini_opinion']=gemini_second_opinion_reviewer(s['patient_id'],s['xray_result'],s['clinical_result'],s['genomic_result'],s['rag_evidence'],s['gpt_opinion']); return s
@timed_node('claude_safety_reviewer')
def node_claude(s): s['claude_safety_opinion']=claude_safety_reviewer(s['patient_id'],s['xray_result'],s['clinical_result'],s['genomic_result'],s['rag_evidence'],s['gpt_opinion'],s['gemini_opinion']); return s
@timed_node('debate_agent')
def node_debate(s):
    xp=float(s['xray_result'].get('tb_probability',0)); cp=float(s['clinical_result'].get('clinical_tb_risk',0)); gc=s['genomic_result'].get('resistance_class','unknown')
    if abs(xp-cp)>=0.35 or s['genomic_result'].get('predicted_class_id',0)>0:
        s['debate_summary']=f'Disagreement or resistance concern detected. RAG and LLM council used. X-ray={xp}, clinical={cp}, genomic={gc}.'
    else:
        s['debate_summary']=f'Broad agreement between image and clinical signals. RAG used for grounding. X-ray={xp}, clinical={cp}, genomic={gc}.'
    return s
@timed_node('judge_agent')
def node_judge(s):
    xs=float(s['xray_result'].get('tb_probability',0)); cs=float(s['clinical_result'].get('clinical_tb_risk',0)); score=round(0.55*xs+0.45*cs,4)
    tb='Likely TB' if score>=0.65 else 'Possible TB' if score>=0.35 else 'Low suspicion / Non-TB'
    gid=int(s['genomic_result'].get('predicted_class_id',0))
    rc='Possible pre-XDR concern / MDR-TB with fluoroquinolone resistance' if gid==2 else 'Possible MDR-TB or drug-resistance concern' if gid==1 else 'No major genomic resistance marker detected'
    s['judge_decision']={'tb_category':tb,'resistance_category':rc,'combined_tb_score':score,'confidence':'high' if score>=0.75 or score<=0.25 else 'medium','human_in_loop_required':True,'doctor_recommendation':'Decision support only. Confirm with physician review, sputum testing, molecular testing, culture, imaging review, and drug-susceptibility testing.'}
    return s
@timed_node('report_generator')
def node_report(s):
    ev='\n'.join([f"### Evidence {i+1}: {e.get('source','unknown')}\n{e.get('content','')}" for i,e in enumerate(s['rag_evidence'])])
    s['final_report']='# TB-Guard Multi-Agent Diagnostic Report\n\n## Patient ID\n'+s['patient_id']+'\n\n## Final Judge Decision\n```json\n'+json.dumps(s['judge_decision'],indent=2)+'\n```\n\n## Clinical input\n```json\n'+json.dumps(s['clinical_data'],indent=2)+'\n```\n\n## Genomic input\n```json\n'+json.dumps(s['genomic_data'],indent=2)+'\n```\n\n## X-ray Agent\n```json\n'+json.dumps(s['xray_result'],indent=2)+'\n```\n\n## Clinical Agent\n```json\n'+json.dumps(s['clinical_result'],indent=2)+'\n```\n\n## WHO Genomic/DNA Agent\n```json\n'+json.dumps(s['genomic_result'],indent=2)+'\n```\n\n## RAG Evidence\n'+ev+'\n\n## GPT Reviewer\n'+s['gpt_opinion']+'\n\n## Gemini Reviewer\n'+s['gemini_opinion']+'\n\n## Claude Safety Reviewer\n'+s['claude_safety_opinion']+'\n\n## Debate Summary\n'+s['debate_summary']+'\n\n## Human-in-the-loop notice\nThis is not a real medical diagnosis. A qualified clinician must confirm all findings.'
    return s

g=StateGraph(TBGuardState)
for name,fn in [('xray',node_xray),('clinical',node_clinical),('genomic',node_genomic),('rag',node_rag),('gpt',node_gpt),('gemini',node_gemini),('claude',node_claude),('debate',node_debate),('judge',node_judge),('report',node_report)]: g.add_node(name,fn)
g.set_entry_point('xray')
for a,b in [('xray','clinical'),('clinical','genomic'),('genomic','rag'),('rag','gpt'),('gpt','gemini'),('gemini','claude'),('claude','debate'),('debate','judge'),('judge','report')]: g.add_edge(a,b)
g.add_edge('report',END)
tbguard_graph=g.compile()
print('LangGraph compiled')

## 10. Demo cases

In [ ]:
def filter_features(data,features): return {f:data.get(f,np.nan) for f in features}
def make_clinical_case(level):
    if level=='low': data={'age':24,'sex':'Female','test_method':'Sputum smear','hiv_education':'No','district':'Ngaliyan','report_month':7,'report_quarter':3}
    elif level=='medium': data={'age':41,'sex':'Female','test_method':'Molecular rapid test','hiv_education':'Yes','district':'Mijen','report_month':7,'report_quarter':3}
    else: data={'age':56,'sex':'Male','test_method':'Molecular rapid test','hiv_education':'Yes','district':'Tugu','report_month':11,'report_quarter':4}
    return filter_features(data,clinical_features)
def make_genomic_case(kind):
    base={g:0 for g in genomic_features}
    if kind=='mdr': base.update({'rpoB':1,'katG':1})
    elif kind=='pre_xdr': base.update({'rpoB':1,'katG':1,'gyrA':1})
    return base
DEMO_CASES={'Patient 1 - Low TB Suspicion':{'patient_id':'P001','xray_image_path':img_paths[0],'clinical_data':make_clinical_case('low'),'genomic_data':make_genomic_case('none')},'Patient 2 - Likely Drug-Sensitive TB':{'patient_id':'P002','xray_image_path':img_paths[1],'clinical_data':make_clinical_case('medium'),'genomic_data':make_genomic_case('none')},'Patient 3 - Possible MDR/pre-XDR TB':{'patient_id':'P003','xray_image_path':img_paths[2],'clinical_data':make_clinical_case('high'),'genomic_data':make_genomic_case('pre_xdr')}}
print(json.dumps(DEMO_CASES,indent=2))

## 11. Run functions

In [ ]:
def initialize_state(case):
    return {'patient_id':case['patient_id'],'xray_image_path':case['xray_image_path'],'clinical_data':case['clinical_data'],'genomic_data':case['genomic_data'],'xray_result':{},'clinical_result':{},'genomic_result':{},'rag_query':'','rag_evidence':[],'gpt_opinion':'','gemini_opinion':'','claude_safety_opinion':'','debate_summary':'','judge_decision':{},'final_report':'','workflow_logs':[],'structured_logs':[]}
def save_run_logs(st):
    path=LOG_DIR/f"{st.get('patient_id','unknown')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_logs.json"
    path.write_text(json.dumps({'patient_id':st.get('patient_id'),'judge_decision':st.get('judge_decision',{}),'workflow_logs':st.get('workflow_logs',[]),'structured_logs':st.get('structured_logs',[]),'final_report':st.get('final_report','')},indent=2),encoding='utf-8')
    return str(path)
def run_tbguard_case(case_name):
    final_state=tbguard_graph.invoke(initialize_state(DEMO_CASES[case_name]))
    final_state['log_path']=save_run_logs(final_state)
    return final_state
def run_all_cases(): return {name:run_tbguard_case(name) for name in DEMO_CASES}
_test=run_tbguard_case(list(DEMO_CASES.keys())[0])
print('\n'.join(_test['workflow_logs']))
print(json.dumps(_test['judge_decision'],indent=2))

## 12. Gradio app

In [ ]:
def gradio_run_single(case_name):
    out = run_tbguard_case(case_name)

    logs = "\n".join(out["workflow_logs"])

    rag_md = ""
    for i, e in enumerate(out["rag_evidence"], 1):
        rag_md += f"### Evidence {i}: {e.get('source', 'unknown')}\n"
        rag_md += f"{e.get('content', '')}\n\n"

    return (
        out["xray_image_path"],
        logs,
        json.dumps(out["xray_result"], indent=2),
        json.dumps(out["clinical_result"], indent=2),
        json.dumps(out["genomic_result"], indent=2),
        rag_md,
        out["gpt_opinion"],
        out["gemini_opinion"],
        out["claude_safety_opinion"],
        out["debate_summary"],
        json.dumps(out["judge_decision"], indent=2),
        out["final_report"],
        out["log_path"]
    )


def gradio_run_all():
    outs = run_all_cases()

    summary = ["# TB-Guard 3-Patient Summary\n"]
    reports = []

    for name, out in outs.items():
        jd = out["judge_decision"]

        summary.append(
            "\n".join([
                "## " + name,
                "",
                "Patient ID: " + out["patient_id"],
                "",
                "TB Category: " + str(jd.get("tb_category")),
                "",
                "Resistance Category: " + str(jd.get("resistance_category")),
                "",
                "Combined TB Score: " + str(jd.get("combined_tb_score")),
                "",
                "Human-in-loop Required: " + str(jd.get("human_in_loop_required")),
                ""
            ])
        )

        reports.append("\n\n# " + name + "\n\n" + out["final_report"])

    p = EXPORT_DIR / f"TBGuard_All_Cases_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    p.write_text("\n".join(reports), encoding="utf-8")

    return "\n".join(summary), "\n".join(reports), str(p)


def normalize_uploaded_file_path(file_obj):
    """
    Handles Gradio file objects across versions.
    file_obj can be a string path, dict, or temp file object.
    """
    if file_obj is None:
        return None

    if isinstance(file_obj, str):
        return file_obj

    if isinstance(file_obj, dict):
        return file_obj.get("path") or file_obj.get("name")

    if hasattr(file_obj, "name"):
        return file_obj.name

    return str(file_obj)


def run_tbguard_custom_case(case):
    """
    Runs the existing LangGraph workflow on a custom case dictionary.
    This lets uploaded X-rays replace the default demo X-rays.
    """
    final_state = tbguard_graph.invoke(initialize_state(case))
    final_state["log_path"] = save_run_logs(final_state)
    return final_state


def gradio_run_uploaded_images(uploaded_files):
    """
    Accepts 1, 2, or 3 uploaded X-ray images.
    Uses the existing Patient 1/2/3 clinical and genomic data templates,
    but replaces the X-ray path with the uploaded image.
    """
    if uploaded_files is None:
        raise gr.Error("Please upload at least one X-ray image.")

    if not isinstance(uploaded_files, list):
        uploaded_files = [uploaded_files]

    uploaded_files = uploaded_files[:3]

    if len(uploaded_files) == 0:
        raise gr.Error("Please upload at least one X-ray image.")

    base_case_names = list(DEMO_CASES.keys())
    outputs = {}
    gallery_items = []

    for i, file_obj in enumerate(uploaded_files):
        image_path = normalize_uploaded_file_path(file_obj)

        if image_path is None:
            continue

        base_name = base_case_names[i]
        base_case = DEMO_CASES[base_name].copy()

        custom_case = {
            "patient_id": f"UPLOAD_P{i+1:03d}",
            "xray_image_path": image_path,
            "clinical_data": base_case["clinical_data"],
            "genomic_data": base_case["genomic_data"]
        }

        case_display_name = f"Uploaded Patient {i+1}"
        outputs[case_display_name] = run_tbguard_custom_case(custom_case)
        gallery_items.append(image_path)

    if len(outputs) == 0:
        raise gr.Error("No valid image files were found.")

    summary_lines = ["# Uploaded X-ray Demo Summary\n"]
    full_reports = []
    combined_logs = []

    for case_name, out in outputs.items():
        jd = out["judge_decision"]

        summary_lines.append(
            "\n".join([
                f"## {case_name}",
                "",
                f"Patient ID: {out['patient_id']}",
                "",
                f"X-ray file: `{out['xray_image_path']}`",
                "",
                f"TB Category: {jd.get('tb_category')}",
                "",
                f"Resistance Category: {jd.get('resistance_category')}",
                "",
                f"Combined TB Score: {jd.get('combined_tb_score')}",
                "",
                f"Human-in-loop Required: {jd.get('human_in_loop_required')}",
                ""
            ])
        )

        combined_logs.append(f"\n\n# {case_name} Logs\n")
        combined_logs.append("\n".join(out["workflow_logs"]))

        full_reports.append(f"\n\n# {case_name}\n\n")
        full_reports.append(out["final_report"])

    report_path = EXPORT_DIR / f"TBGuard_Uploaded_Xray_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    report_path.write_text("\n".join(full_reports), encoding="utf-8")

    log_path = LOG_DIR / f"TBGuard_Uploaded_Xray_Logs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    log_path.write_text("\n".join(combined_logs), encoding="utf-8")

    return (
        gallery_items,
        "\n".join(summary_lines),
        "\n".join(combined_logs),
        "\n".join(full_reports),
        str(report_path),
        str(log_path)
    )


with gr.Blocks(title="TB-Guard Multi-Agent TB Demo") as demo:
    gr.Markdown(
        """
# TB-Guard: Multi-Agent Explainable AI for Tuberculosis

CXR model + clinical model + WHO genomic/DNA expert + FAISS RAG + LangGraph + LLM council/fallback mode.

This is a decision-support demo only. It is not a medical diagnostic system.
"""
    )

    with gr.Tab("Single Patient Workflow"):
        dd = gr.Dropdown(
            choices=list(DEMO_CASES.keys()),
            value=list(DEMO_CASES.keys())[0],
            label="Select patient case"
        )

        btn = gr.Button("Run TB-Guard Workflow")

        with gr.Row():
            xray_img = gr.Image(type="filepath", label="X-ray image used")
            logs_box = gr.Textbox(lines=14, label="Workflow logs")

        with gr.Accordion("Model / Expert Outputs", open=True):
            xray_box = gr.Code(language="json", label="X-ray Agent")
            clinical_box = gr.Code(language="json", label="Clinical Agent")
            genomic_box = gr.Code(language="json", label="WHO Genomic/DNA Agent")

        rag_box = gr.Markdown(label="RAG Evidence")

        with gr.Accordion("LLM Council", open=True):
            gpt_box = gr.Markdown(label="GPT Reviewer")
            gemini_box = gr.Markdown(label="Gemini Reviewer")
            claude_box = gr.Markdown(label="Claude Safety Reviewer")

        debate_box = gr.Markdown(label="Debate Summary")
        judge_box = gr.Code(language="json", label="Judge Decision")
        report_box = gr.Markdown(label="Full Report")
        log_file = gr.File(label="Download structured logs")

        btn.click(
            gradio_run_single,
            dd,
            [
                xray_img,
                logs_box,
                xray_box,
                clinical_box,
                genomic_box,
                rag_box,
                gpt_box,
                gemini_box,
                claude_box,
                debate_box,
                judge_box,
                report_box,
                log_file
            ]
        )

    with gr.Tab("Upload 1–3 X-ray Images"):
        gr.Markdown(
            """
Upload **1, 2, or 3 chest X-ray images**.
The app will run one workflow per uploaded image, up to a maximum of 3.

For uploaded images, the app reuses the existing Patient 1/2/3 clinical and genomic demo templates, but replaces the X-ray with your uploaded image.
"""
        )

        uploaded_xrays = gr.File(
            label="Upload 1 to 3 X-ray images",
            file_count="multiple",
            file_types=["image"]
        )

        run_uploaded_btn = gr.Button("Run Uploaded X-ray Workflow")

        uploaded_gallery = gr.Gallery(label="Uploaded X-rays Used")
        uploaded_summary = gr.Markdown(label="Uploaded Image Summary")
        uploaded_logs = gr.Textbox(lines=16, label="Uploaded Image Workflow Logs")
        uploaded_reports = gr.Markdown(label="Uploaded Image Full Reports")
        uploaded_report_file = gr.File(label="Download Uploaded Image Report")
        uploaded_log_file = gr.File(label="Download Uploaded Image Logs")

        run_uploaded_btn.click(
            gradio_run_uploaded_images,
            uploaded_xrays,
            [
                uploaded_gallery,
                uploaded_summary,
                uploaded_logs,
                uploaded_reports,
                uploaded_report_file,
                uploaded_log_file
            ]
        )

    with gr.Tab("Run All 3 Demo Patients"):
        run_all_btn = gr.Button("Run all demo cases")
        summary_box = gr.Markdown(label="3-patient summary")
        all_reports_box = gr.Markdown(label="All full reports")
        all_reports_file = gr.File(label="Download all reports")

        run_all_btn.click(
            gradio_run_all,
            [],
            [summary_box, all_reports_box, all_reports_file]
        )

demo.launch(share=True)